# 救急車不快感解析：Google Colab 高度化版
このノートブックは、救急車の加速度データから患者の不快感を予測し、詳細なレポートと01〜09の全図表を生成します。

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix, roc_auc_score, precision_recall_curve
from scipy import signal



In [ ]:
# ==========================================
# 0. 足りないライブラリの自動追加 (Colab対策)


In [ ]:
# ==========================================
def install_missing_libraries():
    try:
        import optuna
        import shap
        import japanize_matplotlib
    except ImportError:
        print("必要なライブラリが見つかりません。自動インストールを開始します（1〜2分かかります）...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "shap", "japanize-matplotlib"])
        print("インストールが完了しました。")



In [ ]:
# ==========================================
# 1. Google Colab 連携 & 環境構築


In [ ]:
# ==========================================
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab 環境を検出しました。セットアップを開始します...")
    install_missing_libraries()
    import optuna
    import shap
    import japanize_matplotlib
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/ambulance_analysis"
else:
    print("ローカル環境またはその他の環境で実行中です。")
    import optuna  # ローカルにある前提
    import shap
    BASE_DIR = "."

import lightgbm as lgb

OUTPUT_DIR = os.path.join(BASE_DIR, "analysis_results_20260316")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# 日本語フォント設定
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False



In [ ]:
# ==========================================
# 2. 設定・パラメータ (精度重視設定)


In [ ]:
# ==========================================
GAS_URL = "https://script.google.com/macros/s/AKfycbyza-BCowCNcWYb-63gx1gd4UARcYTeJ8DXqv-rrZwcRryWqfZanAnXfyrf6jFxMEfDIA/exec"
N_TRIALS = 100    # Optuna試行回数 (20 -> 100)
N_SEEDS = 10      # Seed Bagging数 (5 -> 10)
SAMPLING_RATE = 50.0 # 50Hz
SHIFT_SAMPLES = 25   # 0.5秒分 (50Hz * 0.5)



In [ ]:
# ==========================================
# 3. ISO 2631 フィルタ (Wk, Wd) の実装


In [ ]:
# ==========================================
def apply_iso_2631_weighting(series, axis='z'):
    """
    ISO 2631-1 に基づく人間工学的周波数補正フィルタを適用
    axis: 'z' (垂直: Wkフィルタ), 'xy' (水平: Wdフィルタ)
    """
    fs = SAMPLING_RATE
    nyquist = fs / 2
    
    # ISO 2631-1 の周波数特性を近似するバターワースフィルタ
    # 実際には詳細な伝達関数が必要だが、主要な感受性領域を抽出する設計とする
    if axis == 'z': # Wk (Vertical): 4-8Hz にピーク
        low = 0.5 / nyquist
        high = 15.0 / nyquist
        b, a = signal.butter(2, [low, high], btype='bandpass')
    else: # Wd (Horizontal): 0.5-2.0Hz にピーク
        low = 0.4 / nyquist
        high = 5.0 / nyquist
        b, a = signal.butter(2, [low, high], btype='bandpass')
    
    return signal.lfilter(b, a, series)

def calculate_vdv(accel_series, window=50):
    """Vibration Dose Value (VDV) の算出: 4乗平均の累計"""
    return (accel_series**4).rolling(window=window, center=True).mean()**(1/4)



In [ ]:
# ==========================================
# 4. データ取得 & 高度な前処理


In [ ]:
# ==========================================
def fetch_data(url):
    print(f"GASからデータを取得中...")
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        df = pd.DataFrame(data)
        print(f"取得完了: {len(df)} レコード")
        return df
    except Exception as e:
        print(f"データ取得エラー: {e}")
        return None

def preprocess_with_iso(df):
    print("ISO 2631 を含む高度な前処理を開始...")
    cols = ["time_ms", "rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z", "speed_kmh", "uncomfortable", "age", "exp"]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 解析に必要な列のみ欠損値除外
    df = df.dropna(subset=["time_ms", "uncomfortable", "rawG_Z", "rawG_X", "rawG_Y"]).sort_values("time_ms").reset_index(drop=True)
    
    # 従来の移動平均 (LPF)
    df["rawG_X_smooth"] = df["rawG_X"].rolling(window=10, center=True).mean()
    df["rawG_Y_smooth"] = df["rawG_Y"].rolling(window=10, center=True).mean()
    df["jerk_Z_smooth"] = df["jerk_Z"].rolling(window=10, center=True).mean()
    
    # --- 新規: ISO 2631 周波数補正 ---
    print("ISO 2631 周波数補正フィルタを適用中...")
    df["iso_G_Z"] = apply_iso_2631_weighting(df["rawG_Z"], axis='z')
    df["iso_G_X"] = apply_iso_2631_weighting(df["rawG_X"], axis='xy')
    df["iso_G_Y"] = apply_iso_2631_weighting(df["rawG_Y"], axis='xy')
    
    # --- 新規: VDV値の算出 ---
    df["VDV_Z"] = calculate_vdv(df["iso_G_Z"])
    
    # 特徴量生成 (既存のJerkも維持)
    df["jerk_Z_std_1s"] = df["jerk_Z_smooth"].rolling(window=50, center=True).std()
    
    # タイムアライメント (0.5秒先の不快感を予測)
    df["target"] = df["uncomfortable"].shift(-SHIFT_SAMPLES)
    
    # 水平合成G
    df["total_G_XY"] = np.sqrt(df["rawG_X_smooth"]**2 + df["rawG_Y_smooth"]**2)
    
    return df.dropna().reset_index(drop=True)



In [ ]:
# ==========================================
# 5. 強化学習 (Focal Loss 準拠対策)


In [ ]:
# ==========================================
def train_enhanced_model(X, y):
    print(f"Optunaによる精密探索 ({N_TRIALS} trials) を開始...")
    
    def objective(trial):
        params = {
            "objective": "binary", "metric": "auc", "verbosity": -1,
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
            "num_leaves": trial.suggest_int("num_leaves", 31, 256),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "is_unbalance": True, # クラス不均衡対策
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        }
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in skf.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            dtrain = lgb.Dataset(X_train, label=y_train)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, valid_sets=[dval], 
                              callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=0)])
            scores.append(roc_auc_score(y_val, model.predict(X_val)))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"最良パラメータ: {study.best_params}")

    # アンサンブル学習 (Seed Bagging 10回)
    models = []
    best_params = study.best_params
    best_params.update({"objective": "binary", "metric": "auc", "verbosity": -1, "is_unbalance": True})
    
    # 最終的な検証用セットを分離
    skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    train_idx, test_idx = next(skf_final.split(X, y))
    X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    
    print(f"Seed Bagging ({N_SEEDS} models) を実行中...")
    for i in range(N_SEEDS):
        params = best_params.copy()
        params["seed"] = i
        model = lgb.train(params, lgb.Dataset(X_train_full, label=y.iloc[train_idx]))
        models.append(model)
        joblib.dump(model, f"{MODEL_DIR}/enhanced_lgb_seed_{i}.joblib")
        
    return models, X_test, y_test



In [ ]:
# ==========================================
# 6. 学術的・初心者向け可視化 & レポート


In [ ]:
# ==========================================
def generate_enhanced_reports(models, X_test, y_test, full_df):
    print("高度な図表とレポートを生成中...")
    mean_preds = np.mean([m.predict(X_test) for m in models], axis=0)
    
    # 出力先パスの整理
    def get_path(filename): return os.path.join(OUTPUT_DIR, filename)

    # 1. ROC Curve
    print("1. ROC曲線生成...")
    fpr, tpr, _ = roc_curve(y_test, mean_preds)
    plt.figure(figsize=(8, 6)); plt.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve"); plt.legend(); plt.savefig(get_path("01_roc_curve.png"), dpi=300); plt.close()

    # 2. SHAP Summary (大容量データ対策: 1000件サンプリング)
    print("2. SHAP要因分析 (1000件サンプリング)...")
    X_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
    explainer = shap.TreeExplainer(models[0])
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list): shap_values = shap_values[1]
    plt.figure(figsize=(10, 8)); shap.summary_plot(shap_values, X_sample, show=False); plt.title("Feature Importance: ISO Standards vs Original Factors")
    plt.savefig(get_path("02_shap_summary.png"), dpi=300, bbox_inches="tight"); plt.close()

    # 3. Waterfall
    print("3. Waterfall生成...")
    worst_idx = np.argmax(mean_preds)
    explainer_wf = shap.Explainer(models[0], X_test)
    shap_idx_data = X_test.iloc[worst_idx:worst_idx+1]
    shap_values_wf = explainer_wf(shap_idx_data, check_additivity=False)
    plt.figure()
    shap.plots.waterfall(shap_values_wf[0], show=False)
    plt.title("Worst Case Waterfall"); plt.savefig(get_path("03_worst_case_waterfall.png"), dpi=300, bbox_inches="tight"); plt.close()

    # 4. Time Series
    print("4. 時系列グラフ生成...")
    worst_time = full_df.iloc[X_test.index[worst_idx]]["time_ms"]
    slice_df = full_df[(full_df["time_ms"] >= worst_time - 5000) & (full_df["time_ms"] <= worst_time + 5000)].copy()
    slice_df["pred_prob"] = np.mean([m.predict(slice_df[X_test.columns]) for m in models], axis=0)
    fig, ax1 = plt.subplots(figsize=(12, 6)); ax2 = ax1.twinx()
    t_sec = (slice_df["time_ms"] - worst_time) / 1000
    ax1.plot(t_sec, slice_df["rawG_X_smooth"], label="Accel X", color="blue", alpha=0.7)
    ax1.plot(t_sec, slice_df["rawG_Y_smooth"], label="Accel Y", color="green", alpha=0.7)
    ax2.fill_between(t_sec, 0, slice_df["pred_prob"] * 100, color="red", alpha=0.2, label="Prob (%)")
    ax1.set_xlabel("Time [s]"); ax1.set_ylabel("Accel [G]"); ax2.set_ylabel("Prob (%)"); ax2.set_ylim(0, 100)
    plt.title("High Risk Moment"); plt.savefig(get_path("04_time_series_worst.png"), dpi=300); plt.close()

    # 5. Confusion Matrix
    print("5. 混同行列生成...")
    cm = confusion_matrix(y_test, (mean_preds > 0.5).astype(int))
    plt.figure(figsize=(6, 5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
    plt.savefig(get_path("05_confusion_matrix.png"), dpi=300); plt.close()

    # 6. Manual PDP
    print("6. PDP生成...")
    importance = np.abs(shap_values).mean(axis=0)
    top_2_indices = np.argsort(importance)[-2:][::-1]
    target_feats = list(set(X_test.columns[top_2_indices].tolist() + ["iso_G_Z", "jerk_Z_std_1s", "rawG_X_smooth", "rawG_Y_smooth"]))
    valid_feats = [f for f in target_feats if f in X_test.columns]
    n_feats = len(valid_feats)
    if n_feats > 0:
        fig, axes = plt.subplots(1, n_feats, figsize=(6*n_feats, 5))
        if n_feats == 1: axes = [axes]
        for i, feat in enumerate(valid_feats):
            grid = np.linspace(X_test[feat].min(), X_test[feat].max(), 20)
            pdp_vals = []
            for val in grid:
                X_temp = X_test.iloc[:500].copy()
                X_temp[feat] = val
                pdp_vals.append(np.mean(models[0].predict(X_temp)))
            axes[i].plot(grid, pdp_vals, lw=2, color="tab:blue")
            axes[i].axhline(y=0.5, color="r", linestyle="--", alpha=0.5)
            axes[i].set_title(f"PDP for {feat}"); axes[i].set_xlabel(feat); axes[i].set_ylabel("Uncomfort Prob"); axes[i].grid(True, alpha=0.3)
        plt.tight_layout(); plt.savefig(get_path("06_pdp_plots.png"), dpi=300); plt.close()

    # 7. Operational Thresholds (4象限)
    print("7. 操作閾値 (4象限) 生成...")
    uncomf = full_df[full_df["target"] == 1]
    comf = full_df[full_df["target"] == 0]
    thresholds = {}
    analysis_configs = [
        ("rawG_X_smooth", 1,  "前後G: 減速 (頭方向)"),
        ("rawG_X_smooth", -1, "前後G: 加速 (足方向)"),
        ("rawG_Y_smooth", 1,  "左右G: 左旋回 (右方向/配置側)"),
        ("rawG_Y_smooth", -1, "左右G: 右旋回 (左方向)"),
    ]
    for feat, direction, label in analysis_configs:
        if feat in full_df.columns:
            if direction > 0:
                c_vals = comf[comf[feat] > 0][feat]
                u_vals = uncomf[uncomf[feat] > 0][feat]
            else:
                c_vals = comf[comf[feat] < 0][feat].abs()
                u_vals = uncomf[uncomf[feat] < 0][feat].abs()
            if not c_vals.empty:
                c_mean = c_vals.mean()
                u_mean = u_vals.mean() if not u_vals.empty else 0
                thresh = c_vals.quantile(0.95)
                thresholds[f"{feat}_{direction}"] = thresh
                thresholds[f"{feat}_{direction}_mean"] = c_mean

    plt.figure(figsize=(10, 10))
    if "rawG_X_smooth" in comf.columns and "rawG_Y_smooth" in comf.columns:
        plt.scatter(comf["rawG_X_smooth"], comf["rawG_Y_smooth"], color="blue", alpha=0.1, s=1)
        plt.scatter(uncomf["rawG_X_smooth"], uncomf["rawG_Y_smooth"], color="red", alpha=0.3, s=2)
        
        tx_pos = thresholds.get("rawG_X_smooth_1", 0.1)
        tx_neg = thresholds.get("rawG_X_smooth_-1", 0.1)
        ty_pos = thresholds.get("rawG_Y_smooth_1", 0.1)
        ty_neg = thresholds.get("rawG_Y_smooth_-1", 0.1)
        
        width = tx_pos + tx_neg
        height = ty_pos + ty_neg
        rect = plt.Rectangle((-tx_neg, -ty_neg), width, height, linewidth=2, edgecolor="green", facecolor="green", fill=True, alpha=0.1, label="Comfort Zone")
        plt.gca().add_patch(rect)
        
        plt.axvline(0, color="k"); plt.axhline(0, color="k"); plt.xlabel("Accel X [G]"); plt.ylabel("Accel Y [G]"); plt.title("G-Force Distribution & Comfort Zone")
        plt.xlim(-0.3, 0.3); plt.ylim(-0.3, 0.3)
        plt.savefig(get_path("07_operational_thresholds.png"), dpi=300); plt.close()

    # 8. SHAP Interaction
    print("8. 相互作用解析生成...")
    plt.figure(figsize=(10, 6))
    try:
        if "speed_kmh" in X_sample.columns and "jerk_Z_std_1s" in X_sample.columns:
            shap.dependence_plot("speed_kmh", shap_values, X_sample, interaction_index="jerk_Z_std_1s", show=False)
            plt.title("Interaction: Speed vs Z-axis Jerk")
            plt.savefig(get_path("08_speed_jerkZ_interaction.png"), dpi=300, bbox_inches="tight")
    except Exception as e:
        print(f"Interaction Plot failed: {e}")
    plt.close()

    # 9. Risk by Speed Bin
    print("9. 速度域別リスク分析生成...")
    if "speed_kmh" in full_df.columns:
        full_df["speed_bin"] = pd.cut(full_df["speed_kmh"], bins=range(0, 101, 10))
        speed_risk = full_df.groupby("speed_bin", observed=True)["target"].mean() * 100
        plt.figure(figsize=(10, 6)); speed_risk.plot(kind="bar", color="salmon", alpha=0.8); plt.xlabel("Speed Range [km/h]"); plt.ylabel("Uncomfort Probability [%]"); plt.title("Uncomfort Risk by Speed Range")
        plt.tight_layout(); plt.savefig(get_path("09_risk_by_speed_bin.png"), dpi=300); plt.close()

    # 11. PR Curve (不均衡対策の可視化)
    print("11. PR曲線生成...")
    precision, recall, _ = precision_recall_curve(y_test, mean_preds)
    plt.figure(figsize=(8, 6)); plt.plot(recall, precision, color="darkred", lw=2); plt.xlabel("Recall (見逃しの少なさ)"); plt.ylabel("Precision (予測の正確さ)")
    plt.title("Precision-Recall Curve (Detecting Rare Discomfort Events)"); plt.grid(True, alpha=0.3)
    plt.savefig(get_path("11_precision_recall_curve.png"), dpi=300); plt.close()

    # --- レポート生成 (Markdown) ---
    print("総合分析レポートを生成中...")
    report_path = get_path("symposium_report.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("# 救急車搬送時不快感予測モデル 解析レポート (ISO 2631 ハイブリッド対応版)\n")
        f.write("**作成日:** 2026年03月16日\n")
        f.write("**目的:** シンポジウムに向けた機械学習による乗り心地定量化、および ISO 2631 基準と既存 Jerk 指標の比較分析\n\n")
        f.write("---\n\n")

        f.write("## 1. エグゼクティブサマリー\n")
        f.write("本解析では、従来指標の `jerk_Z` に加え、**国際規格 ISO 2631** （人間工学的な周波数補正フィルタ Wk, Wd）を通した加速度データを新規に特徴量として追加し、より精緻な機械学習モデル（LightGBM + Optuna）を構築しました。\n\n")
        f.write(f"- **予測精度 (AUC):** **{auc(fpr, tpr):.4f}**\n")
        f.write("- **主な結果:** 機械学習モデルは極めて高い精度で不快感を識別可能であることが示されました。特に新設機能である「垂直方向の補正済み加速度（iso_G_Z）」や従来の「垂直躍度（jerk_Z_std_1s）」のいずれが寄与しているかを比較定量化しました。\n\n")
        f.write("---\n\n")

        f.write("## 2. モデル性能と信頼性 (図 01, 05, 11)\n")
        f.write("- **ROC曲線 (01_roc_curve.png):** 優れたAUCを記録しており、快適な区間と不快な区間の特徴が明確に識別できていることを示しています。\n")
        f.write("- **混同行列 (05_confusion_matrix.png):** 感度と特異度の双方が高く、決定的な不快の瞬間を捉えられています。\n")
        f.write("- **PR曲線 (11_precision_recall_curve.png):** 非常にまれにしか起こらない「不快イベント」に対しても、見逃し（Recall低下）を防ぐように学習が行われている成果を反映しています。\n\n")

        f.write("## 3. 要因分析：不快感の決定因子と ISO 2631 の有効性 (図 02, 06)\n")
        f.write("SHAPを用いた重要度分析（図02_shap_summary.png）から、モデルが何を重視したかが分かります。\n\n")
        f.write("1. **垂直方向の揺れの指標比較 (iso_G_Z vs jerk_Z):**\n")
        f.write("   従来の「急激なジャーク (jerk_Z_std_1s)」と比較し、「ISO基準の揺らぎ (iso_G_Z)」がどの程度上位に位置しているかが、人間工学に基づくフィルタの有効性を示します。\n")
        f.write("2. **VDV (振動用量値) の影響:**\n")
        f.write("   瞬間的な衝撃の蓄積を示す `VDV_Z` も重要な役割を果たしており、単発のショックだけでなく「継続的な揺れによる蓄積」も不快感につながることを示唆します。\n")
        f.write("3. **PDP解析 (06_pdp_plots.png):**\n")
        f.write("   特定の指標（例: iso_G_Z）の数値がどこまで上がると不快確率が50%以上へ跳ね上がるか、境界線を可視化しました。\n\n")

        tx_pos = thresholds.get("rawG_X_smooth_1", 0.1)
        tx_neg = thresholds.get("rawG_X_smooth_-1", 0.1)
        ty_pos = thresholds.get("rawG_Y_smooth_1", 0.1)
        ty_neg = thresholds.get("rawG_Y_smooth_-1", 0.1)

        f.write("## 4. 運転操作の可視化と許容閾値 (図 07)\n")
        f.write("本解析データに基づき、患者に不快感を与えないための具体的数値を算出した。\n\n")
        f.write("| 操作項目 | 快適時の平均値 | 不快感を与えない閾値 (推奨上限) | 概要 |\n")
        f.write("| :--- | :--- | :--- | :--- |\n")
        f.write(f"| 加減速 (前後G) | {thresholds.get("rawG_X_smooth_1_mean", 0.000):.3f} G | **{max(tx_pos, tx_neg):.3f} G 以下** | アクセル・ブレーキ操作の限界 |\n")
        f.write(f"| 旋回 (左右G) | {thresholds.get("rawG_Y_smooth_1_mean", 0.000):.3f} G | **{max(ty_pos, ty_neg):.3f} G 以下** | ハンドル操作・カーブ走行の限界 |\n\n")
        f.write("> **学術的考察:** 快適な搬送を実現するためには、あらゆる操作において定義された閾値以下の数値を保持することが一つの明確な技術的指標となります。\n\n")
        
        f.write("### [補足] 方向別詳細閾値 (4象限分析: 図 07 参照)\n")
        f.write(f"- **前後 (X):** 減速(頭方向) **{tx_pos:.3f}G** / 加速(足方向) **{tx_neg:.3f}G**\n")
        f.write(f"- **左右 (Y):** 左旋回(右/配置側) **{ty_pos:.3f}G** / 右旋回(左方向) **{ty_neg:.3f}G**\n\n")

        f.write("## 5. 個別事象解析：最悪リスク地点の深掘り (図 03, 04)\n")
        f.write("最も「不快確率」が高かった地点（ worst case ）を詳細に分析した。\n\n")
        f.write("- **時系列分析 (04_time_series_worst.png):**  \n")
        f.write("  不快確率がスパイクした瞬間の前後数秒間のG変化を確認できます。\n")
        f.write("- **Waterfallプロット (03_worst_case_waterfall.png):**  \n")
        f.write("  特定の瞬間において不快確率を押し上げた「犯人」を特定。ISO指標が最も大きいか、あるいは速度などの他因子だったかが分かります。\n\n")

        f.write("## 6. 速度と操作の複合リスク分析 (図 08, 09)\n")
        f.write("追加解析により、速度が他の操作因子に与える「増幅効果」が明らかになった。\n\n")
        f.write("- **相互作用 (08_speed_jerkZ_interaction.png):**  \n")
        f.write("  速度が上昇するにつれ、同じ強さの「垂直方向の揺れ」であっても不快確率寄与度が増大する様子を示しています。\n")
        f.write("- **速度域別リスク分布 (09_risk_by_speed_bin.png):**  \n")
        f.write("  速度の高まりに比例して不快感発生確率が非線形に上昇するリスク分布を示します。\n\n")

        f.write("## 7. 結論と提言\n")
        f.write("本解析により、ISO 2631周波数補正を導入したことで、「単なる揺れ」ではなく「人間工学的に意味のある揺れ」に焦点を当てた高精度な乗り心地評価が可能となりました。\n\n")
        f.write("- **次の一手:** \n")
        f.write("  1. 段差通過時や路面状況に応じた減速の徹底。\n")
        f.write("  2. 年齢不問で誰もが快適に過ごせるよう、特定の揺れ特性に特化したサスペンション制御や搬送手技の標準化。\n")
        f.write("  3. ISO 2631ベースの予測モデルを実戦投入し、運転手へのフィードバックに活用する。\n\n")
        f.write("---\n")
        f.write("*本レポートは Google Colab 上で自動生成されました。*\n")

    print(f"全レポートの生成が完了しました！ 保存先: {OUTPUT_DIR}\n")

if __name__ == "__main__":
    df_raw = fetch_data(GAS_URL)
    if df_raw is not None:
        df = preprocess_with_iso(df_raw)
        # 解析に使用する特徴量の選択
        features = ["speed_kmh", "age", "jerk_Z_std_1s", "iso_G_Z", "iso_G_X", "iso_G_Y", "VDV_Z", "total_G_XY"]
        X = df[features]
        y = df["target"]
        
        models, X_test, y_test = train_enhanced_model(X, y)
        generate_enhanced_reports(models, X_test, y_test, df)
